Using Disjoint Set Union (Union Find) data structure, we can build a random (labelled) tree of $n$ vertices by choosing 2 vertices at random and connecting them via an edge iff they are in different connected components. We do this till $n-1$ edges are added. 
This ensures that the resulting graph does not have a cycle and hence is a tree.

The probability of choosing 2 vertices (with replacement) which from the same component when $e$ edges have been added already is at most $\dfrac{e^2 + n-e}{n^2}$ which is realized when there is only one component having all the $e$ edges. Thus probability of adding $e+1$ th edge is at most $\dfrac{(n-e)(n+e+1)}{n^2}$, which say that probability of adding $e+1$ th edge follows $Geom(\dfrac{(n-e)(n+e+1)}{n^2})$ in the worst case. Thus the expected time for running time for adding the $e+1$ th edge is bounded above by $\dfrac{n^2}{(n-e)(n+e+1)} \leq \dfrac{n^2}{(n-e)^2}$.

Thus the running time for random tree generation of $n$ vertices using DSU is $\mathcal{O}(n^2)$.

In [56]:
# Random Tree with n vertices
# Implementation using Disjoint Set Union (DSU)
# aka Union Find Data Structure
from random import randint
def random_tree_dsu(n):
    dep = [1] * (n+1)
    par = [i for i in range(n+1)]
    adj = [[] for _ in range(n+1)]
    def rep(p):
        if par[p] == p:
            return p
        else:
            par[p] = rep(par[p])
            return par[p]
    edge_cnt = 0
    def merge(p1, p2):
        r1, r2 = rep(p1), rep(p2)
        if r1 != r2:
            if dep[r2] > dep[r1]:
                r1, r2 = r2, r1
            par[r2] = r1
            adj[a].append(b)
            adj[b].append(a)
            return 1
        return 0
    while edge_cnt != n-1:
        a, b = randint(1, n), randint(1, n)
        edge_cnt += merge(a, b)
    return adj

We can use the bijection between Prufer sequences of length $n-2$ and labelled trees of $n$ vertices to get a better algorithm for random (labelled) tree generation.
We generate a random Prufer sequence of length $n-2$ and output the tree corresponding to it. Since this requires us to keep track of the vertex with the highest value (label) and with the minimum remaining degree (edges remaining to be attached), we can use priority queue or heaps. Thus the running time for random tree generation $n$ vertices using Prufer sequences is $\mathcal{O}(nlogn)$.

In [ ]:
# Random tree with n vertices generated using random Prufer code/sequence
from heapq import *
from random import randint
def random_tree_prufer(n):
    cnt = [1] * (n+1)
    pru_code = []
    for _ in range(n-2):
        pru_code.append(randint(1, n))
        cnt[pru_code[-1]] += 1
    adj = [[] for _ in range(n+1)]
    heap = [(cnt[i], -i) for i in range(1, n+1)]
    heapify(heap)
    edge_cnt = 0
    while edge_cnt < n-2:
        c, v = heappop(heap)
        if cnt[-v] != c: continue
        u = pru_code[edge_cnt]
        adj[u].append(-v)
        adj[-v].append(u)
        edge_cnt += 1
        cnt[u] -= 1
        cnt[-v] -= 1
        heappush(heap, (cnt[u], -u))
    u = cnt[2:].index(1) + 2
    adj[1].append(u)
    adj[u].append(1)
    return adj

We can build a random (labelled) tree by keeping only one connected component adding the remaining vertices randomly to a random vertex in the connected component. This approach is clearly $\mathcal{O}(n)$ and since we need to add $n-1$ edges, we can't get a better running time.

Since there is no random element picking for sets in python, we keep the remaining vertices (to be added) in a list and keep the indices where they are present in another list (dictionary if the vertices are not sequential). When a vertex from the remaining list is selected, we swap that vertex with the vertex that is at the end of the list. We update the index list accordingly. Then the last vertex in the list (which is the selected vertex) is removed from the list (to exclude it from further considerations).

In [ ]:
# Random tree generation using a custom data structure 
# (list + set using list with dictionary (list in this case as 
# the vertices are sequential 1 to n) for index keeping)
from random import choice
def random_tree_modset(n):
    adj = [[] for _ in range(n+1)]
    added = [1]
    rem = []
    rem_ind = [n-1] * (n+1)
    for i in range(n-1):
        rem.append(i+2)
        rem_ind[i+2] = i
    for i in range(n-2, -1, -1):
        u = choice(added)
        v = choice(rem)
        rem_ind[rem[i]] = rem_ind[v]
        rem[i], rem[rem_ind[v]] = rem[rem_ind[v]], rem[i] 
        rem.pop()
        added.append(v)
        adj[u].append(v)
        adj[v].append(u)
    return adj

The difference in tree generation time of the above 3 algorithms can be clearly seen below.

In [77]:
tree_dsu = random_tree_dsu(5 * 10**5)

In [79]:
tree_prufer = random_tree_prufer(5 * 10**5)

In [80]:
tree_modset = random_tree_modset(5 * 10**5)

The following code uses DFS to compute the optimal number of troupes neededed to be stationed for a given tree. Since the call stack is memory heavy and python has very low recursion depth by default, usage of this code is only recommended when $n$ is of the order of 100's or even better, never.

In [59]:
def min_troupe_rec(adj, n=None):
    if n == None: n = len(adj)-1
    opt = [0] * (n+1)
    dc = [0] * (n+1)
    with1 = [0] * (n+1)
    with2 = [0] * (n+1)
    # Post order using DFS on tree
    def dfs(v, p):
        for nei in adj[v]:
            if nei != p:
                dfs(nei, v)
        tot_child_opt = sum(opt[u] for u in adj[v] if u!=p)
        with2[v] = sum(dc[u] for u in adj[v] if u!=p) + 2
        with1[v] =  tot_child_opt + 1
        with0 = n
        for u in adj[v]:
            if u==p: continue
            with0 = min(with0, tot_child_opt - opt[u] + with2[u])
        opt[v] = min(with1[v], with2[v], with0)
        dc[v] = min(opt[v], tot_child_opt)
        return
    dfs(1, 0)
    return opt[1]

The following code uses stack explicitly to mimick the recursive call stack. This is memory efficient (compared to the recursive version) and allows for faster computation of optimal stationing of the troupes even when $n$ is in millions.

In [60]:
def min_troupe(adj, n=None):
    if n == None: n = len(adj)-1
    opt = [0] * (n+1)
    dc = [0] * (n+1)
    with1 = [0] * (n+1)
    with2 = [0] * (n+1)
    # DFS using stack
    stack = [(1, 0, 0)]
    while stack:
        v, p, c = stack.pop()
        if c == 0:
            stack.append((v, p, 1))
            for nei in adj[v]:
                if nei!=p: stack.append((nei, v, 0))
        else:
            tot_child_opt = sum(opt[u] for u in adj[v] if u!=p)
            with2[v] = sum(dc[u] for u in adj[v] if u!=p) + 2
            with1[v] =  tot_child_opt + 1
            with0 = n
            for u in adj[v]:
                if u==p: continue
                with0 = min(with0, tot_child_opt - opt[u] + with2[u])
            opt[v] = min(with1[v], with2[v], with0)
            dc[v] = min(opt[v], tot_child_opt)
    return opt[1]

In [67]:
n = 4 * 10 ** 6
tree = random_tree_modset(n)
# print(tree)
print(min_troupe(tree, n))

2482499


Due the structure of the trees involved, all non-isomorphic trees are not choosen at random. This causes the average percentage to differ from the one where non-isomorphic trees were chosen at random. For example, there are 2 non-isomorphic trees for $n=4$ (star and path) with minimum troupe count 2 and 3 respectively. Thus, the average percentage has to be 62.5\%. But the simulating one gets around 68.75\%. This is because there are $n$ possible numberings for a star while $\frac{n!}{2}$ numberings for a path.

In [62]:
# The average number of troupes required seems to be around 62% of the number of towns (n)
trials = 10 ** 3
total = 0
for _ in range(trials):
    n = randint(1, 10**4)
    total += (min_troupe(random_tree_dsu(n), n))/n
print(total/trials)

0.6213080303054584


In [63]:
trials = 10 ** 3
total = 0
for _ in range(trials):
    n = randint(1, 10**4)
    total += min_troupe(random_tree_modset(n), n) / n
print(total / trials)

0.620824258897574


In [64]:
n = 10 ** 4
trials = 10 ** 3
total = 0
for _ in range(trials):
    total += min_troupe(random_tree_prufer(n), n)
print(total / trials / n)

0.6209236
